In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from lmfit import Model

plt.rcParams['figure.figsize'] = (12,6)
plt.rcParams['font.size'] = 14


In [2]:
def load_scans(txt_folder, num_files=None):
    files = sorted([f for f in os.listdir(txt_folder) if f.endswith(".txt")])
    if num_files is not None:
        files = files[:num_files]

    df_list = []

    for f in files:
        path = os.path.join(txt_folder, f)
        df = pd.read_csv(path, header=5, sep=",", usecols=[0,1])
        df.columns = ["time", "ampl"]
        df_list.append(df)

    print(f"Loaded {len(df_list)} scans.")
    return df_list


In [3]:
txt_folder = "2fexp10_2"   # uprav podle potřeby
df_scans = load_scans(txt_folder)

# spojíme časové osy a amplitudy do jednoho dataframe
df_all = pd.concat(df_scans, axis=1)

# oddělíme čas a amplitudu
time_cols = [c for c in df_all.columns if "time" in c]
ampl_cols = [c for c in df_all.columns if "ampl" in c]

df_time = df_all[time_cols]
df_ampl = df_all[ampl_cols]

print(df_time.shape, df_ampl.shape)


Loaded 1602 scans.


MemoryError: Unable to allocate 122. MiB for an array with shape (1602, 10001) and data type float64

In [ ]:
df_time_mean = df_time.mean(axis=1)
df_ampl_mean = df_ampl.mean(axis=1)

df_avg = pd.DataFrame({
    "time": df_time_mean,
    "ampl": df_ampl_mean
})

df_avg.to_csv("averaged.txt", index=False)
df_avg.head()


In [ ]:
df_mask = pd.read_csv("goutput_1.csv")
x_mask = df_mask["x"].values
y_mask = df_mask["y"].values


In [ ]:
def fit_single_peak(x_exp, y_exp, x_mask, y_mask):
    y_mask_interp = np.interp(x_exp, x_mask, y_mask)

    def model_func(x, P, a, b):
        return P * y_mask_interp + a*x + b

    model = Model(model_func)
    params = model.make_params(P=1.0, a=0.0, b=0.0)

    result = model.fit(y_exp, params, x=x_exp)

    return result.params["P"].value


In [ ]:
def extract_peak_window(x_full, y_full, x_mask):
    xmin, xmax = x_mask.min(), x_mask.max()
    mask = (x_full >= xmin) & (x_full <= xmax)
    return x_full[mask], y_full[mask]


In [ ]:
def concentration_from_P(P, CO2_ref=10.0):
    return P * CO2_ref


In [ ]:
def allan_variance(data):
    N = len(data)
    avar = []
    taus = []

    for m in [1, 2, 4, 8, 16, 32, 64]:
        if 2*m >= N:
            break
        taus.append(m)
        diffs = data[2*m:] - 2*data[m:-m] + data[:-2*m]
        avar.append(0.5 * np.mean(diffs**2))

    return np.array(taus), np.array(avar)


In [ ]:
averaging_lengths = [15, 30, 60, 120, 240, 480, 960]

concentration_series = {}

for L in averaging_lengths:
    conc_list = []

    for i in range(0, len(df_avg), L):
        block = df_avg.iloc[i:i+L]
        if len(block) < L:
            continue

        x_block = block["time"].values
        y_block = block["ampl"].values

        x_fit, y_fit = extract_peak_window(x_block, y_block, x_mask)
        if len(x_fit) < 10:
            continue

        P = fit_single_peak(x_fit, y_fit, x_mask, y_mask)
        conc = concentration_from_P(P)

        conc_list.append(conc)

    concentration_series[L] = np.array(conc_list)
    print(f"Averaging {L} scans → {len(conc_list)} concentration values")


In [ ]:
allan_results = {}

for L, conc_data in concentration_series.items():
    taus, avar = allan_variance(conc_data)
    allan_results[L] = (taus, avar)


In [ ]:
plt.figure(figsize=(12,6))

for L, (taus, avar) in allan_results.items():
    plt.loglog(taus, np.sqrt(avar), marker='o', label=f"{L} scans")

plt.xlabel("Averaging time τ (blocks)")
plt.ylabel("Allan deviation σ(τ)")
plt.title("Allanova odchylka koncentrace CO₂")
plt.legend()
plt.grid(True, which="both", ls="--")
plt.tight_layout()
plt.show()
